# Member 1 - CUB-200-2011 EDA experiments (Task 1.1)

This notebook is the **experiment record** for the data work. Per spec section 5.4 the final code is pure
Python: the same analysis runs end to end with

```
python -m src.cli eda
```

which writes every figure and table to `outputs/eda/`. The cells below call the same functions step by step
so each finding can be inspected interactively. Run from the project root or from `notebooks/`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd.parent) if (p / 'src' / 'cli.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the project root or its notebooks/ folder.')
sys.path.insert(0, str(ROOT))

from src.utils.config import load_config, output_dir
from src.data.hf import load_cub, dataset_class_names
from src.data import audit, captions
from src.data.split import load_split
from src.data.loaders import build_loaders, verify_loader
from src.data.transforms import build_transform

cfg = load_config('configs/data.yaml')
OUT = output_dir(cfg, 'eda', 'notebook')

## 1. Load the original dataset
The official test split stays separate from all model selection.

In [ ]:
data = load_cub(cfg)
train, test = data['train'], data['test']
names = dataset_class_names(train, test)
print('Original train:', len(train), '| Official test:', len(test), '| Species:', len(names))

## 2. Class distribution and image quality
Flags (very dark, low contrast, smaller than 224px, non-RGB) are for human review, **not** automatic exclusion.

In [ ]:
counts = pd.Series(train['label']).value_counts().sort_index()
counts.plot(kind='bar', figsize=(17, 3), title='Training images per species', xticks=[])
plt.show()
print(counts.describe())

meta = audit.scan_images(train, 'train')
flags = audit.quality_flags(meta)
print('Decode failures:', (~meta.decode_ok).sum(), '| Flagged images:', len(flags))
print(flags.flag_reason.str.split('; ').explode().value_counts())
audit.contact_sheets(train, flags, OUT / 'review')

## 3. Caption cleaning
Split the ten newline-separated captions, normalise whitespace and case, remove repeated captions and
fragments shorter than three words (e.g. the lone caption `this`). Typos are reported, not auto-corrected.

In [ ]:
result = captions.analyse_captions(train, 'train', OUT)
print(result['summary'])
captions.typo_candidates(result['tokens']).head(20)

## 4. Fixed stratified train/validation split
The team manifest (`data/splits/train_validation_split.csv`, 80/20 stratified, seed 42) is reused, never regenerated.

In [ ]:
import numpy as np
labels = np.asarray(train['label'])
train_ids, val_ids, split_hash = load_split(cfg, labels)
print('Train:', len(train_ids), '| Validation:', len(val_ids), '| sha256:', split_hash[:16], '...')
print('Species in each split:', len(set(labels[train_ids])), len(set(labels[val_ids])))

## 5. Exact-duplicate leakage check
Hash decoded pixels of every train and test image; a duplicate spanning splits would leak information.

In [ ]:
test_meta = audit.scan_images(test, 'test')
split_of = {**{i: 'train' for i in train_ids}, **{i: 'validation' for i in val_ids}}
report = audit.duplicate_report(pd.concat([meta, test_meta]), split_of)
print('Duplicate groups:', report['exact_duplicate_groups'], '| cross-split:', report['cross_split_groups'])

## 6. Preprocessing, augmentation and full DataLoader checks
All models use 224x224 inputs. Training augmentation recipes: `transfer_mild` (pretrained models) and
`scratch_strong` (scratch CNN). Evaluation: resize to 256, center-crop 224. Every batch is checked.

In [ ]:
print('scratch_strong:', build_transform(224, train=True, recipe='scratch_strong'))
print('evaluation:', build_transform(224, train=False))
loaders = build_loaders(cfg, data, augmentation='scratch_strong')
for name, loader, n in (('train', loaders.train, len(train_ids)), ('validation', loaders.validation, len(val_ids)),
                        ('test', loaders.test, len(test))):
    print(name, verify_loader(loader, n, loaders.image_size))